# 10 · Lo que vino después
*De InstructGPT a DPO y a Constitutional AI: qué pieza del pipeline cambia cada uno*

Notebook del episodio 10 de **InstructGPT** ([ver la lección](https://d32vh5r3pvn9et.cloudfront.net/cursos/instructgpt/lo-que-vino-despues)). Es el mismo guion del video, en texto, con cada bloque de código para correr y su salida real.

**Mecanismo:** qué pieza cambia cada receta → modelos en memoria (4 vs 2) → π* ∝ π_SFT e^{r/β} → r = β log π/π_SFT + β log Z → L_DPO (Z se cancela) → DPO en el juguete de ep09: γ=0 oro 5,62 / capacidad 0,27; γ=0,1 oro 5,11 / 0,975 → win rate oro vs SFT: PPO-ptx 0,73, DPO 0,65; DPO vs PPO 0,41 → r implícita ordena 0,915 (RM explícito 0,945) → DPO paper 61 % vs 57 % (juez GPT-4) → CAI: SL (crítica→revisión→SFT) + RL (IA elige → PM → RLAIF) → juez IA en el juguete: 86,6 %, 46 % de etiquetas extremas → recorte 40–60 % y entropía cruzada blanda → qué sobrevivió → las seis series → volver

**Cómo correrlo:** con el entorno de `~/Movies/serie-instructgpt/lab` (`cd lab && uv run --project ~/Movies/serie-alineacion/lab --group notebooks --with jupyterlab jupyter lab ../notebooks`, o en VS Code eligiendo el kernel `lab/.venv`). Todo corre offline: Gemma 2 2B sale de la caché de Hugging Face, y los entrenamientos son de juguete, con datos sintéticos.

Cada bloque de código arranca de cero (`%reset -f`): la celda de **preparación** es lo que el video no muestra, y la siguiente es exactamente el código que aparece en pantalla.

## ▶ Antes de empezar

Corré la celda de abajo una vez: baja el código de la serie y sus dependencias. Funciona en Google Colab (gratis) y en cualquier Jupyter. Para guardar tus cambios en Colab: **Archivo → Guardar una copia en Drive**.

In [ ]:
import os, subprocess, sys
SERIE = "instructgpt"
REPO = "https://github.com/Faridmurzone/paperclase-notebooks"

def _raiz():  # carpeta del repo si ya estamos adentro
    d = os.path.abspath(os.getcwd())
    while True:
        if os.path.isfile(os.path.join(d, ".paperclase-notebooks")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

RAIZ = _raiz()
if RAIZ is None:
    RAIZ = os.path.abspath("paperclase-notebooks")
    if not os.path.isdir(RAIZ):
        subprocess.run(["git", "clone", "--depth", "1", REPO, RAIZ], check=True)
os.chdir(os.path.join(RAIZ, SERIE, "lab"))  # los bloques de código importan desde snippets/ y leen datos/
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements.txt"], check=True)
os.environ["HF_HUB_OFFLINE"] = "0"  # el notebook original corría sin red; acá baja los modelos de Hugging Face
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("listo:", os.getcwd())

In [1]:
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import warnings; warnings.filterwarnings("ignore")

*InstructGPT. Episodio diez, el último: lo que vino después. El pipeline de este paper tiene tres pasos, SFT, supervised fine-tuning, reward model y PPO, proximal policy optimization, y cuatro modelos en memoria. Después aparecieron dos ideas que cambian piezas distintas: DPO, direct preference optimization, que saca el reward model y el aprendizaje por refuerzo; y constitutional AI, o CAI, que reemplaza parte de las etiquetas humanas por etiquetas de una IA. Las medimos sobre el mismo juguete del episodio nueve.*

### Qué pieza cambia cada receta

El mapa del episodio. El SFT sigue en las tres recetas; en CAI, las demostraciones salen de respuestas que el modelo critica y corrige. Quién compara: en DPO siguen siendo personas; en CAI, para lo dañino, compara una IA guiada por principios escritos. El reward model: DPO no lo entrena aparte, queda escondido en la política. Y la optimización: DPO reemplaza PPO por una pérdida de clasificación; CAI mantiene el aprendizaje por refuerzo.

| pieza | InstructGPT | DPO | CAI |
|---|---|---|---|
| SFT | demos humanas | el mismo SFT | respuestas revisadas por la IA |
| quién compara | personas | personas | IA con principios (daño) |
| reward model | aparte, 6B | implícito en la política | aparte, con etiquetas de IA |
| optimización | PPO-ptx | una pérdida de clasificación | PPO |

## Parte 1 · DPO: sin reward model ni refuerzo

La derivación completa está en Alineación, episodio cinco. Acá nos quedamos con qué cambia en el pipeline y cuánto rinde.

### Cuántos modelos hay que tener en memoria

Primero, el costo de cada receta. InstructGPT entrena con cuatro modelos a la vez: la política, el SFT como referencia para el KL, la divergencia de Kullback-Leibler, el reward model, y la value function, que el paper inicializa desde el reward model. DPO necesita dos: la política y la referencia. Y la diferencia más grande: PPO genera texto en cada paso; DPO entrena sobre pares fijos. Cuatro contra dos modelos, con muestreo contra sin muestreo. CAI, en su fase de refuerzo, es como InstructGPT.

In [2]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep10_0_piezas.py
__file__ = 'snippets/ep10_0_piezas.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
# Qué modelos hay que tener en memoria durante el entrenamiento de preferencias, según cada receta.

In [3]:
recetas = {
    "InstructGPT (PPO-ptx)": ["política", "SFT (referencia)", "RM", "value function"],
    "DPO": ["política", "SFT (referencia)"],
    "CAI (RL-CAI)": ["política", "referencia", "PM", "value function"],
}
generan_en_el_loop = {"InstructGPT (PPO-ptx)": True, "DPO": False, "CAI (RL-CAI)": True}
for receta, modelos in recetas.items():
    online = "muestrea en cada paso" if generan_en_el_loop[receta] else "pares fijos, sin muestrear"
    print(f"{receta:22} {len(modelos)} modelos · {online}")

InstructGPT (PPO-ptx)  4 modelos · muestrea en cada paso
DPO                    2 modelos · pares fijos, sin muestrear
CAI (RL-CAI)           4 modelos · muestrea en cada paso


### De la política óptima a la pérdida DPO

$$
\pi^*(y\mid x) = \tfrac{1}{Z(x)}\,\pi^{SFT}(y\mid x)\, e^{\,r(x,y)/\beta}
$$
<small>la solución del objetivo con KL · $Z(x)$: normaliza</small>

$$
r(x,y) = \beta \log \tfrac{\pi^*(y\mid x)}{\pi^{SFT}(y\mid x)} + \beta \log Z(x)
$$
<small>despejar r: tomar log y multiplicar por β</small>

$$
L_{DPO} = -\log \sigma\Big(\beta \log \tfrac{\pi(y_w)}{\pi^{SFT}(y_w)} - \beta \log \tfrac{\pi(y_l)}{\pi^{SFT}(y_l)}\Big)
$$
<small>la ecuación 1 de InstructGPT, con $r$ implícita: $Z$ se cancela</small>

La cuenta, en tres líneas. El objetivo de InstructGPT, recompensa menos beta por KL, tiene una solución exacta: el SFT repesado por e a la r sobre beta, y dividido por Z, que hace que todo sume uno. En el dibujo, cuatro respuestas: los logits suben con la recompensa y el softmax los vuelve probabilidades. Si tomamos logaritmo y multiplicamos por beta, despejamos r: beta por el log del cociente, más un término que solo depende del prompt. En el dibujo, el log-cociente es una suma por token. Metemos esa r en la pérdida de Bradley-Terry de la ecuación uno: el término de Z es igual para las dos respuestas, y se cancela. Queda una pérdida que solo usa la política y el SFT. En el dibujo, la sigmoide del margen.

### DPO sobre los mismos rankings del episodio nueve

En el juguete, con los mismos datos que usó el reward model fuerte. Rankings de cuatro respuestas del SFT. Los seis pares de cada ranking: la elegida y la rechazada. Las log-probabilidades del SFT congelado, la referencia. El margen: cuánto subió la elegida respecto de la referencia, menos cuánto subió la rechazada, por beta. La pérdida es menos el log de la sigmoide del margen. Y le podemos sumar la misma mezcla de preentrenamiento. Sin mezcla, oro cinco coma seis dos, pero la capacidad cae a cero coma dos siete: el impuesto del episodio siete también existe en DPO. Con gamma cero coma uno, oro cinco coma uno uno y capacidad cero coma nueve siete cinco.

In [4]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep10_1_dpo.py
__file__ = 'snippets/ep10_1_dpo.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import copy, importlib.util, itertools, os, torch
import torch.nn.functional as F
_r = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "comun", "igpt_juguete.py")
_s = importlib.util.spec_from_file_location("igpt_juguete", _r); j = importlib.util.module_from_spec(_s); _s.loader.exec_module(j)
sft = j.cargar("ep09_sft.pt").eval()                       # el SFT del episodio 9
K = 4
PARES = list(itertools.combinations(range(K), 2))
gumbel = torch.distributions.Gumbel(0.0, 1.0)
def comparar(n):                                           # las mismas «personas» del episodio 9
    x = j.prompts(n).repeat_interleave(K, 0)
    y = j.muestrear(sft, x).view(n, K, -1)
    gusto = j.oro(y.flatten(0, 1)).view(n, K) + gumbel.sample((n, K))
    return y.gather(1, gusto.argsort(-1, descending=True)[..., None].expand_as(y))
def logp(model, y):                                        # log π(y | x), sumado sobre la respuesta
    return j.logprobs(model, y).sum(-1)

In [5]:
def dpo(gamma, beta=0.5, pasos=300):
    pol = copy.deepcopy(sft).train()
    opt = torch.optim.AdamW(pol.parameters(), lr=1e-4)
    for paso in range(pasos):
        y = comparar(64)                        # rankings como los del RM
        w = torch.cat([y[:, a] for a, b in PARES])   # elegida de cada par
        l = torch.cat([y[:, b] for a, b in PARES])   # rechazada
        with torch.no_grad():
            ref_w, ref_l = logp(sft, w), logp(sft, l)   # referencia
        margen = beta * ((logp(pol, w) - ref_w) - (logp(pol, l) - ref_l))
        loss = -F.logsigmoid(margen).mean()
        loss = loss + gamma * j.nll_pre(pol, j.batch_pre(64))   # ptx
        loss.backward(); opt.step(); opt.zero_grad()
    return pol.eval()

In [6]:
# Verificación extra (corrió junto con el video)
for gamma in [0.0, 0.1]:
    torch.manual_seed(2)
    pol = dpo(gamma)
    y = j.muestrear(pol, j.prompts(512))
    print(f"γ={gamma}  oro {j.oro(y).mean():.2f}  capacidad {j.capacidad(pol):.3f}")
    j.guardar(pol, f"ep10_dpo_{gamma}.pt")

γ=0.0  oro 5.62  capacidad 0.272


γ=0.1  oro 5.11  capacidad 0.975


### DPO contra PPO-ptx, con el oro como juez

La comparación justa. Las dos políticas: el PPO-ptx del episodio nueve y el DPO con mezcla. Los mismos mil prompts de test, y el SFT como baseline. Win rate contra el SFT con dos jueces. Y DPO contra PPO-ptx directo. PPO-ptx: cero coma setenta y tres según el oro; en el episodio nueve dio cero coma setenta y dos, con otra tirada de muestras. DPO: cero coma sesenta y cinco. Mano a mano, DPO gana cero coma cuarenta y uno: en este juguete, PPO-ptx queda adelante. Una razón posible: PPO aprende de sus propias respuestas; DPO solo de pares del SFT.

In [7]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep10_2_winrate.py
__file__ = 'snippets/ep10_2_winrate.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import importlib.util, os, torch
_r = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "comun", "igpt_juguete.py")
_s = importlib.util.spec_from_file_location("igpt_juguete", _r); j = importlib.util.module_from_spec(_s); _s.loader.exec_module(j)
sft = j.cargar("ep09_sft.pt").eval()
rm, _ = j.cargar_rm("ep09_rm_fuerte.pt")
torch.manual_seed(4)
x = j.prompts(1000, torch.Generator().manual_seed(11))      # los mismos prompts de test del episodio 9
def win_rate(a, b):
    return ((a > b).float() + 0.5 * (a == b).float()).mean().item()

In [8]:
politicas = {"PPO-ptx (ep. 9)": j.cargar("ep09_pol_fuerte_0.02.pt"),
             "DPO, γ = 0,1":    j.cargar("ep10_dpo_0.1.pt")}
with torch.no_grad():
    y_sft = j.muestrear(sft, x)
    y = {n: j.muestrear(p.eval(), x) for n, p in politicas.items()}
    for n in politicas:
        print(f"{n:16} vs SFT: juez RM {win_rate(rm(y[n]), rm(y_sft)):.2f}"
              f"  juez oro {win_rate(j.oro(y[n]), j.oro(y_sft)):.2f}")
    ppo, dpo = y.values()
    print(f"DPO vs PPO-ptx, juez oro: {win_rate(j.oro(dpo), j.oro(ppo)):.2f}")

PPO-ptx (ep. 9)  vs SFT: juez RM 0.77  juez oro 0.73
DPO, γ = 0,1     vs SFT: juez RM 0.66  juez oro 0.65
DPO vs PPO-ptx, juez oro: 0.41


### Win rate según el oro

En barras. PPO-ptx contra el SFT. DPO contra el SFT: mejora clara, con dos modelos en memoria en vez de cuatro y sin generar texto durante el entrenamiento. DPO contra PPO-ptx: debajo del empate. La referencia del cero coma cinco. Es un juguete con un solo beta y un learning rate: no dice cuál es mejor en general. Dice que la comparación depende de la tarea y del ajuste.

- **PPO-ptx vs SFT**: 0,73
- **DPO vs SFT**: 0,65
- **DPO vs PPO-ptx**: 0,41
- **empate**: 0,5

### La política es un reward model

El título del paper de DPO dice que tu modelo de lenguaje es, en secreto, un reward model. Lo comprobamos. La recompensa implícita de una respuesta: beta por el log del cociente entre la política DPO y el SFT. Los mismos pares nuevos que usamos para medir el reward model del episodio nueve. ¿Ordena bien? El reward model explícito, cero coma nueve cuatro cinco. La recompensa implícita, cero coma nueve uno cinco, sin haber entrenado ningún reward model.

**Cada línea, su término:**

- `def r_implicita(y):` → $\hat r(x,y) = \beta \log \tfrac{\pi_{DPO}(y\mid x)}{\pi^{SFT}(y\mid x)}$ — sin $\beta \log Z(x)$: es igual para $a$ y $b$
- `beta * (j.logprobs(dpo, y) - j.logprobs(sft, y)).sum(-1)` → $\beta \textstyle\sum_t \big[\log \pi_{DPO}(y_t) - \log \pi^{SFT}(y_t)\big]$ — una suma por token
- `distintos = oa != ob` → $\text{oro}(a) \ne \text{oro}(b)$ — solo los pares donde el oro elige
- `(r(a) > r(b)) == (oa > ob)` → $\mathbf{1}\big[\hat r(a) > \hat r(b)\big] = \mathbf{1}\big[\text{oro}(a) > \text{oro}(b)\big]$ — el acierto es el promedio

In [9]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep10_3_implicita.py
__file__ = 'snippets/ep10_3_implicita.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import importlib.util, os, torch
_r = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "comun", "igpt_juguete.py")
_s = importlib.util.spec_from_file_location("igpt_juguete", _r); j = importlib.util.module_from_spec(_s); _s.loader.exec_module(j)
sft = j.cargar("ep09_sft.pt").eval()
dpo = j.cargar("ep10_dpo_0.1.pt").eval()
rm, sesgo = j.cargar_rm("ep09_rm_fuerte.pt")
g = torch.Generator().manual_seed(7)
x = j.prompts(1000, g)
torch.manual_seed(7)
a, b = j.muestrear(sft, x), j.muestrear(sft, x)               # los mismos pares nuevos del episodio 9

In [10]:
beta = 0.5
@torch.no_grad()
def r_implicita(y):                         # β · log π_DPO(y|x) / π_SFT(y|x)
    return beta * (j.logprobs(dpo, y) - j.logprobs(sft, y)).sum(-1)

oa, ob = j.oro(a), j.oro(b)
distintos = oa != ob                        # pares donde el oro elige
for nombre, r in [("RM explícito", rm), ("r implícita DPO", r_implicita)]:
    with torch.no_grad():
        acierto = ((r(a) > r(b)) == (oa > ob))[distintos].float().mean()
    print(f"{nombre:16} ordena bien {acierto:.3f} de los pares")

RM explícito     ordena bien 0.945 de los pares
r implícita DPO  ordena bien 0.915 de los pares


> **Rafailov et al., 2023 — Direct Preference Optimization: Your Language Model is Secretly a Reward Model** · [arXiv:2305.18290](https://arxiv.org/abs/2305.18290)
>
> **61 % vs 57 %** · «DPO has a win rate of approximately 61% at a temperature of 0.0, exceeding the performance of PPO at 57% at its optimal sampling temperature of 0.0.»

En el paper usan modelos de hasta seis mil millones de parámetros; esta cifra es de resumir posts de Reddit. Contra resúmenes escritos por personas, DPO gana cerca del sesenta y uno por ciento y PPO el cincuenta y siete. El juez es GPT cuatro, y lo validaron con personas: coincide con ellas más o menos tanto como las personas entre sí. Otro cambio respecto de InstructGPT: un modelo como juez.

## Parte 2 · Constitutional AI: etiquetas de una IA

InstructGPT necesitó decenas de miles de comparaciones humanas. CAI reemplaza las de daño por una lista de principios.

### Constitutional AI, en dos fases

Las dos fases. La supervisada: un asistente que solo fue entrenado para ser útil responde prompts dañinos; el mismo modelo critica su respuesta según un principio sorteado de la constitución, y la revisa. Se repite, y con las respuestas revisadas se hace un SFT. Es el bucle que programamos en Alineación, episodio cinco. La de refuerzo: el modelo de la fase uno genera dos respuestas, y una IA elige la mejor según un principio. Con esas etiquetas se entrena un PM, preference model, que es su nombre para el reward model, y después aprendizaje por refuerzo. Lo llaman RLAIF, reinforcement learning from AI feedback.

prompts dañinos → respuesta → crítica: principio → revisión → SFT: SL-CAI → dos respuestas → IA elige: principio → PM → RL: RL-CAI

> **Bai et al., 2022 — Constitutional AI: Harmlessness from AI Feedback** · [arXiv:2212.08073](https://arxiv.org/abs/2212.08073)
>
> **16 principios** · «The only human oversight is provided through a list of rules or principles, and so we refer to the method as ‘Constitutional AI’.»

La frase del abstract. La única supervisión humana es una lista de reglas o principios: escribieron dieciséis para comparar respuestas. Ojo: la utilidad sigue saliendo de etiquetas humanas; lo que reemplazan con la IA son las de daño. Y dicen que esos principios los eligieron de forma ad hoc, y que en el futuro deberían definirlos más partes interesadas. Es la pregunta abierta de InstructGPT, sección cinco punto dos.

### Un juez IA en el juguete

En CAI, el juez es un modelo de lenguaje al que se le pregunta cuál respuesta sigue mejor un principio, y se leen las probabilidades de la opción A y la B. Nuestro análogo, burdo: el propio SFT, comparando qué tan probable le parece cada respuesta. La sigmoide de la diferencia es una etiqueta blanda: la probabilidad de que A sea mejor, no un cero o un uno. El recorte que usó CAI: llevar cada etiqueta al rango cuarenta a sesenta por ciento. ¿Cuánto acierta? Ochenta y seis coma seis por ciento: sin ninguna persona. Pero el cuarenta y seis por ciento de las etiquetas están casi en cero o en uno: demasiado seguras. El recorte las deja entre cero coma cuatro y cero coma seis.

In [11]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep10_4_juez_ia.py
__file__ = 'snippets/ep10_4_juez_ia.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import importlib.util, os, torch
_r = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "comun", "igpt_juguete.py")
_s = importlib.util.spec_from_file_location("igpt_juguete", _r); j = importlib.util.module_from_spec(_s); _s.loader.exec_module(j)
sft = j.cargar("ep09_sft.pt").eval()
torch.manual_seed(8)
x = j.prompts(2000, torch.Generator().manual_seed(13))
a, b = j.muestrear(sft, x), j.muestrear(sft, x)

In [12]:
@torch.no_grad()
def juez(a, b):                     # la IA compara: log-verosimilitud de cada respuesta
    la = j.logprobs(sft, a).sum(-1)       # el propio SFT hace de juez
    lb = j.logprobs(sft, b).sum(-1)
    return torch.sigmoid(la - lb)         # P(A mejor): etiqueta blanda

p = juez(a, b)
p_rec = p.clamp(0.4, 0.6)                 # recorte 40–60 % de CAI
oa, ob = j.oro(a), j.oro(b)
m = oa != ob
print(f"acierto del juez IA:  {((p > 0.5) == (oa > ob))[m].float().mean():.3f}")
print(f"etiquetas casi 0 o 1: {((p < 0.05) | (p > 0.95)).float().mean():.0%}")
print(f"después del recorte:  p entre {p_rec.min():.1f} y {p_rec.max():.1f}")

acierto del juez IA:  0.866
etiquetas casi 0 o 1: 46%
después del recorte:  p entre 0.4 y 0.6


### Etiquetas blandas y recortadas

$$
p = \sigma(\ell_A - \ell_B)
$$
<small>$\ell$: log-probabilidad que el juez le da a cada opción</small>

$$
\tilde p = \min\!\big(\max(p,\, 0{,}4),\, 0{,}6\big)
$$
<small>el recorte 40–60 % de CAI (con cadena de razonamiento)</small>

$$
L = -\big[\tilde p \log \sigma(\Delta r) + (1 - \tilde p)\log\big(1 - \sigma(\Delta r)\big)\big]
$$
<small>$\Delta r = r(A) - r(B)$ · entropía cruzada con objetivo blando</small>

En fórmulas. l sub A y l sub B son las log-probabilidades que el juez le da a cada opción. La etiqueta blanda es la sigmoide de la diferencia. En el dibujo, tres pares: dos muy seguros y uno en duda. El recorte: si p es menor que cero coma cuatro, vale cero coma cuatro; si es mayor que cero coma seis, vale cero coma seis. CAI lo usó con las etiquetas que salen de una cadena de razonamiento, que eran demasiado extremas. El PM se entrena con entropía cruzada contra esa etiqueta blanda. Con objetivo cero coma seis, la pérdida es mínima cuando sigma de delta r vale cero coma seis, con delta r cero coma cuatro uno: cero coma seis siete. Si el PM se pone más seguro, con delta r igual a dos, sube a cero coma nueve tres. La etiqueta recortada le prohíbe estar tan seguro.

## Parte 3 · Qué quedó de InstructGPT

DPO y CAI cambian piezas. El esqueleto sigue siendo el de este paper.

### Lo que sobrevivió

- Primero un SFT, siempre
- Preferencias por pares: Bradley-Terry
- Un freno contra el SFT: el mismo β
- Evaluar con win rate, con un juez caro y uno barato

Lo que sigue en pie. Todo empieza con un SFT: en DPO es la referencia, y en CAI el punto de partida de las dos fases. Las preferencias se expresan por pares y se modelan con Bradley-Terry, explícito o implícito. El freno de KL contra el SFT: en DPO el beta es literalmente el mismo de la ecuación dos de InstructGPT. Y la evaluación por win rate, con su intervalo de confianza. Lo que cambió es quién juzga: muchas veces, otro modelo. Y el episodio nueve mostró por qué hay que validar a ese juez.

## Parte 4 · Cerrar el ciclo

Con este episodio terminan las seis series. Es buen momento para volver a la primera.

### Las seis series, y a dónde volver

El recorrido completo. Las bases: Deep learning a secas, Álgebra lineal para LLMs y Optimizadores, con foco en AdamW, el Adam, adaptive moment estimation, con weight decay desacoplado. El modelo y cómo se ajusta: Attention is all you need, LoRA, low-rank adaptation, y esta serie. Y a dónde volver. Alineación a bajo nivel fue la primera, y la escribimos sin todo esto debajo. Ahora cada episodio se lee distinto. Y Caja blanca, donde están tus experimentos propios con probes, steering y auditoría.

Deep learning → Álgebra lineal → Optimizadores → Attention → LoRA → InstructGPT → Alineación → Caja blanca

### Volver a Alineación con lo nuevo

- ep. 2 y 3: SFT y RM, ahora con InstructGPT 3, 4 y 9
- ep. 4 y 5: PPO y DPO, con el impuesto y el juez
- ep. 6: Goodhart, que acá viste en una corrida
- ep. 7 a 10 y Caja blanca: mirar adentro del modelo

Una guía para la relectura. Alineación, episodios dos y tres: el SFT y el reward model. Ahora sabés de dónde salieron sus datos y cuánto pesa cada decisión. Episodios cuatro y cinco: PPO y DPO. Ahora con el impuesto de alineación, la mezcla de preentrenamiento y la comparación con dos jueces. Episodio seis: Goodhart y la sobreoptimización, que viste pasar en el juguete del episodio nueve. Y del siete al diez, mirar adentro del modelo: hooks, probes, SAEs, los sparse autoencoders, y supervisión. Ahí entran las series de attention y álgebra, y Caja blanca, para pasar de los mecanismos a tus propios experimentos.

## Resumiendo

- DPO: sin RM aparte ni RL; 2 modelos, pares fijos
- Juguete: DPO 0,65 y PPO-ptx 0,73 contra el SFT (oro)
- La política DPO ordena pares como un RM: 0,915
- CAI: SFT con críticas y revisiones, y RL con etiquetas de IA
- Etiquetas blandas, recortadas a 40–60 %
- Próximo: volver a Alineación a bajo nivel y a Caja blanca